# MDS202-Activities 2–5: Data Wrangling with Pandas
**Dataset:** Kickstarter projects (`kickstarter_projects.csv`)

In [82]:
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 140)
df = pd.read_csv("kickstarter_projects.csv", parse_dates=["Launched", "Deadline"])
df.columns = df.columns.str.strip()

## **Data Profile (Activity 2)**

In [83]:
# First and last records
display(pd.concat([df.head(1), df.tail(1)]))

,ID,Name,Category,Subcategory,Country,Launched,Deadline,Goal,Pledged,Backers,State
0,1860890148,Grace Jones Does Not Give A F$#% T-Shirt (limited Edition),Fashion,Fashion,United States,2009-04-21 21:02:48,2009-05-31,1000,625,30,Failed
374852,1339173863,Spirits of the Forest,Games,Tabletop Games,Spain,2018-01-02 15:02:31,2018-01-26,24274,4483,82,Live


In [84]:
# Number of rows and columns
rows, cols = df.shape
print(f"Rows: {rows:,}\nColumns: {cols}")

Rows: 374,853
Columns: 11


In [85]:
# Group columns by type
num_vars  = df.select_dtypes(include="number").columns.tolist()
date_vars = df.select_dtypes(include="datetime").columns.tolist()
cat_vars  = df.select_dtypes(exclude=["number", "datetime"]).columns.tolist()

variable_summary = pd.DataFrame(
    [(c, "Numerical", str(df[c].dtype)) for c in num_vars]
    + [(c, "Date/Time", str(df[c].dtype)) for c in date_vars]
    + [(c, "Categorical/Text", str(df[c].dtype)) for c in cat_vars],
    columns=["Column", "Variable Group", "Actual Type"],
)
display(variable_summary)

,Column,Variable Group,Actual Type
0,ID,Numerical,int64
1,Goal,Numerical,int64
2,Pledged,Numerical,int64
3,Backers,Numerical,int64
4,Launched,Date/Time,datetime64[us]
5,Deadline,Date/Time,datetime64[us]
6,Name,Categorical/Text,str
7,Category,Categorical/Text,str
8,Subcategory,Categorical/Text,str
9,Country,Categorical/Text,str


In [118]:
# Descriptive statistics
display(df[["Goal", "Pledged", "Backers"]].describe().round(0))

,Goal,Pledged,Backers
count,374853.0,374853.0,374853.0
mean,45864.0,9121.0,107.0
std,1158778.0,91321.0,912.0
min,0.0,0.0,0.0
25%,2000.0,31.0,2.0
50%,5500.0,625.0,12.0
75%,16000.0,4051.0,57.0
max,166361391.0,20338986.0,219382.0


**Description:** I ran describe() to get a quick statistical summary of a DataFrame's numeric columns 

In [87]:
# Columns with missing values
missing = pd.DataFrame({"missing": df.isnull().sum(), "missing_%": (df.isnull().mean() * 100).round(2)})
print("Columns with missing values:")
if (missing["missing"] > 0).any():
    display(missing[missing["missing"] > 0])
else:
    print("None")

Columns with missing values:
None


In [88]:
# Possible identifier columns
n = len(df)
identifier_summary = pd.DataFrame({
    "Unique Count": df.nunique(),
    "Unique %": (df.nunique() / n * 100).round(2),
})
identifier_summary["Looks like identifier (>90%)"] = identifier_summary["Unique %"] > 90
display(identifier_summary)
print("Duplicated IDs:", df["ID"].duplicated().sum(), "| Repeated project names:", df["Name"].duplicated().sum())

,Unique Count,Unique %,Looks like identifier (>90%)
ID,374853,100.00,True
Name,372061,99.26,True
Category,15,0.00,False
Subcategory,159,0.04,False
Country,22,0.01,False
Launched,374297,99.85,True
Deadline,3164,0.84,False
Goal,27692,7.39,False
Pledged,39989,10.67,False
Backers,3963,1.06,False


Duplicated IDs: 0 | Repeated project names: 2792


**Description:** I counted unique values in each column to find possible identifiers, and checked for repeated IDs and names. Only **ID** is a true identifier (100% unique, no repeats). **Name** is 99.26% unique but 2,792 names repeat, so it is a label, not a key. **Launched** is a near-unique timestamp, not an identifier.

In [89]:
# show every unique value of the key categoricals, with repr() so hidden whitespace becomes visible
for col in ["Category", "State", "Country"]:
    print(f"--- {col}: {df[col].nunique()} unique values ---")
    print([repr(v) for v in sorted(df[col].unique())], "\n")

--- Category: 15 unique values ---
["'Art'", "'Comics'", "'Crafts'", "'Dance'", "'Design'", "'Fashion'", "'Film & Video'", "'Food'", "'Games'", "'Journalism'", "'Music'", "'Photography'", "'Publishing'", "'Technology'", "'Theater'"] 

--- State: 5 unique values ---
["'Canceled'", "'Failed'", "'Live'", "'Successful'", "'Suspended'"] 

--- Country: 22 unique values ---
["'Australia'", "'Austria'", "'Belgium'", "'Canada'", "'Denmark'", "'France'", "'Germany'", "'Hong Kong'", "'Ireland'", "'Italy'", "'Japan'", "'Luxembourg'", "'Mexico'", "'Netherlands '", "'New Zealand'", "'Norway'", "'Singapore'", "'Spain'", "'Sweden'", "'Switzerland'", "'United Kingdom'", "'United States'"] 



**Interpretation.** The listing exposes **'Netherlands '** with a trailing space, an inconsistency that a plain **.unique()** display hides.

### **Additional Checks in Data Exploration for Anomalies or Suspicious Data**

In [90]:
print(f"Max Pledged: ${df['Pledged'].max():,}")  
print(f"Min Pledged: ${df['Pledged'].min():,}")  

Max Pledged: $20,338,986
Min Pledged: $0


In [91]:
whitespace_names     = (df["Name"] != df["Name"].str.strip()).sum()
non_ascii    = df["Name"].str.contains(r"[^\x00-\x7F]", regex=True).sum()
repeat_names = df["Name"].duplicated().sum()

print(f"Names with leading/trailing spaces: {whitespace_names:,}")
print(f"Names with non-ASCII characters:    {non_ascii:,}")
print(f"Repeated names:                     {repeat_names:,}")
print("\n")

print(f"Example")
display(df.loc[df["Name"] != df["Name"].str.strip(), "Name"].head(3).map(repr))

Names with leading/trailing spaces: 1,108
Names with non-ASCII characters:    9,531
Repeated names:                     2,792


Example


0             'Grace Jones Does Not Give A F$#% T-Shirt (limited Edition) '
62    'Indie Nerd Board Game, Needs Player Character Miniature Sculpted!  '
83                  'Quixote & Evangenitals to Edinburgh Fringe Festival! '
Name: Name, dtype: str

**Description:** I checked project names for leading/trailing spaces, non-English characters and repeats, and displayed examples.

In [92]:
# Goal: zeros, goals of $10M or more
print("Goal == 0:", (df["Goal"] == 0).sum(), df.loc[df["Goal"] == 0, "State"].value_counts().to_dict())  
print("Goal >= 10,000,000:", (df["Goal"] >= 10_000_000).sum())  

Goal == 0: 4 {'Successful': 2, 'Canceled': 1, 'Failed': 1}
Goal >= 10,000,000: 188


In [93]:
zero_backers_pledge = ((df["Backers"] == 0) & (df["Pledged"] > 0))
backers_zero_pledge = ((df["Backers"] > 0) & (df["Pledged"] == 0))
print("0 backers but Pledged > 0:", zero_backers_pledge.sum())     # 3
print("Backers > 0 but Pledged = 0:", backers_zero_pledge.sum())   # 9
display(df.loc[zero_backers_pledge | backers_zero_pledge, ["Name", "State", "Goal", "Pledged", "Backers"]])

0 backers but Pledged > 0: 3
Backers > 0 but Pledged = 0: 9


,Name,State,Goal,Pledged,Backers
5115,Sins Of A Son,Failed,3500,1,0
185102,iStandUp!,Failed,50000,218,0
282368,The Falcon Fly By,Failed,5000,1,0
317931,Muebles pintados a mano con arte,Failed,14015,0,1
317952,Magical products engraved with drawings /and PRINTS <3,Failed,459,0,1
319989,dailybicho.com,Failed,470,0,1
320904,Calle indie,Failed,22747,0,1
322803,The Nail Bar,Failed,9758,0,1
323464,The Real People of Mexico Photobook & Video Documentary.,Canceled,4932,0,1
323538,Hogar Del Arte,Failed,2466,0,1


**Description:** I looked for rows where Backers and Pledged disagree: no backers but money pledged, or backers but no money.

In [94]:
df.columns = df.columns.str.strip()

# Planned campaign length in days. normalize() drops Launched's time of day so the count isn't short by a day.
df["campaign_days"] = (df["Deadline"] - df["Launched"].dt.normalize()).dt.days

# Not-successful, finished projects. Live is excluded (still ongoing); Goal == 0 is excluded (Pledged / Goal is undefined).
not_success = df[df["State"].isin(["Failed", "Canceled", "Suspended"]) & (df["Goal"] > 0)].copy()
not_success["percent_to_goal"] = (not_success["Pledged"] / not_success["Goal"] * 100).round(2)

# Which "not successful" projects still reached their goal, and in which states?
reached = not_success[not_success["percent_to_goal"] >= 100]
print(reached["State"].value_counts().rename("Projects").to_frame())

cols = ["Name", "Goal", "Pledged", "Backers", "State", "campaign_days", "percent_to_goal"]
failed_but_reached  = reached[reached["State"] == "Failed"][cols]                    # suspicious
stopped_but_reached = reached[reached["State"].isin(["Canceled", "Suspended"])][cols]  # explainable: stopped after reaching the goal
truly_unsuccessful  = not_success[not_success["percent_to_goal"] < 100]

display(failed_but_reached)
display(stopped_but_reached.head())

           Projects
State              
Canceled        697
Suspended       292
Failed            9


,Name,Goal,Pledged,Backers,State,campaign_days,percent_to_goal
2295,The new face of Canadian Science Fiction,10000,10797,66,Failed,59,107.97
5754,"POPE! an epic musical, opening at NY Fringe Festival 2010",3600,3600,30,Failed,30,100.00
25983,From Neurons to Nirvana: The Great Medicines,40000,40000,113,Failed,60,100.00
33823,Music Empowers - A Day with the Accende Ensemble,3000,3088,20,Failed,58,102.93
62467,Tye Gazoo,3000,3000,24,Failed,22,100.00
86441,"""Balance""",23000,23020,141,Failed,30,100.09
108392,Key Wrench - Unlock Your Life!,6000,6183,208,Failed,41,103.05
151769,Egg Salad,10,10,1,Failed,30,100.00
259754,The 5th Annual 90 Square Arts Festival,12000,12000,77,Failed,50,100.00


,Name,Goal,Pledged,Backers,State,campaign_days,percent_to_goal
58,Save the Ramzi (Canceled),1000,1190,18,Canceled,32,119.00
1106,Help Sex Piano Fund an EP (Canceled),750,1110,32,Canceled,77,148.00
3255,Tiny Urban Park (Canceled),650,1260,39,Canceled,14,193.85
6152,Pieces of My Soul: Scattered Around Chicago. (Canceled),60,160,4,Canceled,90,266.67
7604,Microfiche releasing Jetpacks and Laser Beams on vinyl (Canceled),1500,1905,24,Canceled,36,127.00


**Description:** Created campaign_days (campaign length). Then, for Failed, Canceled and Suspended projects with a valid goal, calculated percent_to_goal and listed those that still reached 100% or more. This was the anomaly i detected in the previous activity.

### **Data Dictionary**

In [116]:
info = {
    "ID":          ("Unique numeric identifier of the project",
                    "None; 100% unique"),
    "Name":        ("Project title",
                    "1,108 names have leading/trailing spaces; 9,531 contain non-ASCII characters; 2,792 names repeat"),
    "Category":    ("Top-level category (15 values)",
                    "No issues found"),
    "Subcategory": ("Sub-category within the category (159 values)",
                    "97,268 rows repeat the Category value as the Subcategory - validated below"),
    "Country":     ("Country of the project, as a full name (22 values)",
                    "'Netherlands ' has a trailing space (2,868 rows)"),
    "Launched":    ("Date and time the campaign went live",
                    "Timestamp has time of day, while Deadline has none, so subtracting gives day counts that are off by up to 1"),
    "Deadline":    ("Date the funding window closes (no time of day)",
                    "Latest value is 2018-03-03; Live projects may have a stale state"),
    "Goal":        ("Funding target; the file has no currency column, so the currency is unverified",
                    "4 rows have Goal = 0 (impossible); 188 goals are at least $10M; extremely right-skewed"),
    "Pledged":     ("Amount pledged; same unit as Goal",
                    "Extremely right-skewed (max $20.3M)"),
    "Backers":     ("Number of backers who pledged",
                    "3 rows have 0 backers but a pledge; 9 rows have backers but a 0 pledge"),
    "State":       ("Outcome: Successful, Failed, Canceled, Suspended or Live",
                    "'Live' (2,798) has no final outcome; 989 Canceled/Suspended projects reached their goal"),
}
print('Subcategory == Category rows:', (df['Subcategory'] == df['Category']).sum())     
print('Latest Deadline:', df['Deadline'].max())                                                 

dictionary = pd.DataFrame({
    "Column": list(info),
    "Data Type": [str(df[c].dtype) for c in info],
    "Meaning": [meaning for meaning, _ in info.values()],
    "Potential Issue": [issue for _, issue in info.values()],
})
display(dictionary)

Subcategory == Category rows: 97268
Latest Deadline: 2018-03-03 00:00:00


,Column,Data Type,Meaning,Potential Issue
0,ID,int64,Unique numeric identifier of the project,None; 100% unique
1,Name,str,Project title,"1,108 names have leading/trailing spaces; 9,531 contain non-ASCII characters; 2,792 names repeat"
2,Category,str,Top-level category (15 values),No issues found
3,Subcategory,str,Sub-category within the category (159 values),"97,268 rows repeat the Category value as the Subcategory - validated below"
4,Country,str,"Country of the project, as a full name (22 values)","'Netherlands ' has a trailing space (2,868 rows)"
5,Launched,datetime64[us],Date and time the campaign went live,"Timestamp has time of day, while Deadline has none, so subtracting gives day counts that are off by up to 1"
6,Deadline,datetime64[us],Date the funding window closes (no time of day),Latest value is 2018-03-03; Live projects may have a stale state
7,Goal,int64,"Funding target; the file has no currency column, so the currency is unverified",4 rows have Goal = 0 (impossible); 188 goals are at least $10M; extremely right-skewed
8,Pledged,int64,Amount pledged; same unit as Goal,Extremely right-skewed (max $20.3M)
9,Backers,int64,Number of backers who pledged,3 rows have 0 backers but a pledge; 9 rows have backers but a 0 pledge


### **Reflection**  
**1. What are the most important variables in the dataset?**  
Each of the variables has its own significance and could contribute meaningful interpretations when used. For finding out for the traction of the project and success rate would be **State, Goal, Pledge, Backers**. For grouping projects into categories, **Categories, Subcategory, and Country**, while the **Launch, Deadline** presents the campaign duration.

**2. Which variables require additional investigation before analysis?**  
There are multiple variables that require further evaluation before data processing (all of them are presented in the Additional Checks column. Some of these belong to the following variables:  
1. *Goal* - Has zero and extreme values
2. *Pledge* - Has a max pledge that skews the data further and pledges that also have conflict with the variable **State**
3. *State* - The failed statuses are associated with projects that have reached their goal.
4. *Country* and Name - Have trailing whitespaces.  

**3. What data-quality issues did you identify during exploration?**
I found 4 projects with a goal of 0, a trailing-space inconsistency in the "Country" variable for Netherlands, whitespace in 1,108 project names, a strongly right-skewed goal and pledged, 2,798 Live projects without a final outcome since live values are still ongoing, 3 projects with pledges but no backers and 9 projects without pledges but with a single backer each, and 989 Canceled/Suspended projects whose pledges reached the goal (but i think this can be justified because a creator can cancel after reaching it).

## **Manipulation & Aggregation (Activity 3)**

### Q1. Which categories have the highest success rate among finished projects?
*Live projects are filtered out first because they have no final outcome yet.*

In [96]:
finished = df[df["State"] != "Live"].copy()                       # boolean filtering
finished["Is_Successful"] = finished["State"].eq("Successful")    # derived column

q1 = (
    finished.groupby("Category")
    .agg(
        Projects=("ID", "count"),
        Success_Rate_Pct=("Is_Successful", "mean"),
        Avg_Backers=("Backers", "mean"),
        Median_Pledged=("Pledged", "median"),
    )
    .assign(Success_Rate_Pct=lambda d: (d["Success_Rate_Pct"] * 100).round(1))
    .sort_values("Success_Rate_Pct", ascending=False)
    .reset_index()                                                # category back to a column; clean 0..n-1 index
)
q1["Avg_Backers"] = q1["Avg_Backers"].round(1)
display(q1)
print(f"Overall success rate (finished projects): {finished['Is_Successful'].mean():.1%} of {len(finished):,}")

,Category,Projects,Success_Rate_Pct,Avg_Backers,Median_Pledged
0,Dance,3749,62.4,43.0,1855.0
1,Theater,10870,60.1,47.1,1553.0
2,Comics,10743,54.4,135.3,1502.0
3,Music,49248,48.9,54.9,1003.5
4,Art,27957,41.2,42.3,426.0
5,Film & Video,62362,37.9,67.2,755.5
6,Games,34938,35.8,323.8,1303.0
7,Design,29760,35.4,241.5,1926.0
8,Publishing,39079,31.5,56.8,291.0
9,Photography,10730,30.8,39.7,240.0


Overall success rate (finished projects): 36.0% of 372,055


**Interpretation.**  
Performance and arts categories (Dance 62.4%, Theater 60.1%, Comics 54.4%) succeed far more often than Technology (20.0%), Journalism (21.4%), or Crafts (24.2%), against an overall rate of 36.0%. Games and Design attract the most backers per project (324 and 242) but they succeed only about 35% of the time, which indicates that even when they draw large audiences it does not guarantee success. Using the *median* pledge avoids letting a few multi-million-dollar projects dominate.

### Q2. How have the number of projects and the success rate changed by launch year?
Live projects are filtered out because they have no final outcome yet.

In [97]:
finished = df[df["State"] != "Live"].copy()                        # boolean filtering
finished["Launch_Year"] = finished["Launched"].dt.year             # derived column
finished["Is_Successful"] = finished["State"].eq("Successful")     # derived column

q2 = (
    finished.groupby("Launch_Year")
    .agg(
        Projects=("ID", "count"),
        Success_Rate_Pct=("Is_Successful", "mean"),
        Median_Goal=("Goal", "median"),
    )
    .assign(Success_Rate_Pct=lambda d: (d["Success_Rate_Pct"] * 100).round(1))
    .reset_index()
)
display(q2)

,Launch_Year,Projects,Success_Rate_Pct,Median_Goal
0,2009,1329,43.6,3000.0
1,2010,10519,43.7,3500.0
2,2011,26236,46.4,4000.0
3,2012,41161,43.5,5000.0
4,2013,44836,43.3,6000.0
5,2014,66993,31.5,5772.0
6,2015,74919,28.0,7000.0
7,2016,56535,33.0,7000.0
8,2017,49525,37.3,5708.0
9,2018,2,0.0,2065.0


**Interpretation.**  
The platform grew quickly until 2015, when it had the most finished projects (74,919), and then it slowly declined. Simultaneously, the success rate fell from about 43–46% (2009–2013) to a low of 28.0% in 2015. Furthermore, the median goal rose from 3,000 (2009) to 7,000 (2015), so part of the decline in success is consistent with more ambitious projects. Also, 2018 has only 2 finished projects (the rest are still Live), so its 0.0% is not a real rate and should be excluded from the trend.

### Q3. How does goal size relate to backers, pledged amounts and success?
*Finished projects with a valid goal only.*

In [98]:
bins   = [0, 1_000, 5_000, 10_000, 50_000, 100_000, np.inf]
labels = ["Under $1k", "$1k-$5k", "$5k-$10k", "$10k-$50k", "$50k-$100k", "Over $100k"]

# Rows removed, in the order they are removed
live_rows    = df[df["State"] == "Live"]                  # step 1: live, no final outcome yet
finished     = df[df["State"] != "Live"].copy()
finished["Is_Successful"] = finished["State"].eq("Successful")

invalid_goal = finished[finished["Goal"] <= 0]           # step 2: invalid goal, goal is less than or equal to 0
q3_base      = finished[finished["Goal"] > 0].copy()
q3_base["Goal_Tier"] = pd.cut(q3_base["Goal"], bins=bins, labels=labels, include_lowest=True)

q3 = (
    q3_base.groupby("Goal_Tier", observed=True)
    .agg(
        Projects=("ID", "count"),
        Avg_Backers=("Backers", "mean"),
        Avg_Pledged=("Pledged", "mean"),
        Success_Rate_Pct=("Is_Successful", "mean"),
    )
    .assign(Success_Rate_Pct=lambda d: d["Success_Rate_Pct"] * 100)
    .round(1)
    .reset_index()
)
display(q3)

# Sanity Check
accounting = pd.DataFrame({
    "Step": ["All projects", "Removed: Live (no final outcome)", "Removed: invalid goal (Goal = 0)", "Used in Q3"],
    "Rows": [len(df), -len(live_rows), -len(invalid_goal), q3["Projects"].sum()],
})
display(accounting)
assert len(live_rows) + len(invalid_goal) + q3["Projects"].sum() == len(df)

,Goal_Tier,Projects,Avg_Backers,Avg_Pledged,Success_Rate_Pct
0,Under $1k,60447,24.3,870.9,49.4
1,$1k-$5k,122230,48.3,2540.7,42.8
2,$5k-$10k,65983,101.2,6207.1,35.4
3,$10k-$50k,94599,175.4,16265.7,26.8
4,$50k-$100k,16217,262.2,34980.0,14.0
5,Over $100k,12575,390.5,41661.2,5.8


,Step,Rows
0,All projects,374853
1,Removed: Live (no final outcome),-2798
2,Removed: invalid goal (Goal = 0),-4
3,Used in Q3,372051


**Interpretation.**  
Bigger goals bring bigger audiences but lower success (24 backers and 871USD pledged on average under 1k USD, versus 391 backers and 41,661USD above 100k USD) where it collapses from 49.4% to 5.8%. Bigger ambitions draw bigger audiences without drawing enough money to close the gap. Projects over 100k USD raise only about 42k USD on average That is too far from their goal, so most fail. 

#### **Summary**

| # | Analytical question | Operations used | Key finding |
|---|---|---|---|
| Q1 | Which categories have the highest success rate among finished projects? | Boolean filter, derived flag, `groupby`, 4 aggregations, sort, `reset_index` | Dance (62.4%) and Theater (60.1%) lead; Technology is lowest (20.0%) against an overall 36.0% |
| Q2 | How have the number of projects and the success rate changed by launch year? | Boolean filter, 2 derived columns, `groupby`, 3 aggregations | Volume peaked in 2015 (74,919 finished projects); success fell from 43-46% (2009-2013) to 28.0% in 2015, then recovered to 37.3% in 2017 |
| Q3 | How does goal size relate to backers, pledged amounts and success? | Boolean filter, derived bins (`pd.cut`), `groupby`, 4 aggregations | Success falls from 49.4% (under 1k) to 5.8% (over 100k) |

## **Transformation (Activity 4A)**
Working copy **df_transformed** carries the transformations forward into cleaning.

In [99]:
df_transformed = df.copy()
print("df_transformed shape before transformations:", df_transformed.shape)

df_transformed shape before transformations: (374853, 12)


### **T1. Reshaping with pivot_table()**

In [100]:
pivot_df = df_transformed.pivot_table(values="Pledged", index="Category", columns="State", aggfunc="median")
print("Shape after reshaping  (category-level):", pivot_df.shape)
print("Empty cells in the pivot:", int(pivot_df.isna().sum().sum()))
display(pivot_df.round(0))

Shape after reshaping  (category-level): (15, 5)
Empty cells in the pivot: 0


State,Canceled,Failed,Live,Successful,Suspended
Category,,,,,
Art,21.0,67.0,126.0,2631.0,3.0
Comics,130.0,247.0,730.0,4000.0,20.0
Crafts,10.0,37.0,63.0,1825.0,16.0
Dance,75.0,105.0,259.0,3364.0,10.0
Design,817.0,490.0,1643.0,14623.0,1923.0
Fashion,62.0,70.0,160.0,6864.0,140.0
Film & Video,52.0,105.0,132.0,5210.0,10.0
Food,25.0,81.0,220.0,8952.0,2.0
Games,389.0,222.0,446.0,10572.0,10.0


**What one row represents:**  
In **pivot_df**, one row is one *Category*; each column is a project *State*; each cell is the **median amount pledged** by projects in that category and state. In *df_transformed*, one row is one project, and median replaces mean because pledged amounts are heavily skewed.

### **T2. Launch volume by calendar month and year (count of projects, months ordered Jan→Dec)**

In [101]:
month_order = ["January", "February", "March", "April", "May", "June", "July", "August",
               "September", "October", "November", "December"]
df_transformed["Launch_Year"]  = df_transformed["Launched"].dt.year
df_transformed["Launch_Month"] = df_transformed["Launched"].dt.month
df_transformed["Launch_Month_Name"] = pd.Categorical(df_transformed["Launched"].dt.month_name(), categories=month_order, ordered=True)

launch_table = (
    df_transformed
    .groupby(["Launch_Month_Name", "Launch_Year"], observed=True)
    .size()
    .unstack(fill_value=0)
)
display(launch_table)

fin = df_transformed[df_transformed["State"] != "Live"]
month_success = (fin.groupby("Launch_Month_Name", observed=True)["State"]
                    .apply(lambda s: (s == "Successful").mean() * 100).round(1))
display(month_success.to_frame("Success_Rate_Pct"))

Launch_Year,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
Launch_Month_Name,,,,,,,,,,
January,0,296,1594,2902,3296,3475,6490,4863,4451,124
February,0,371,1803,3358,3359,3817,7363,5242,4027,0
March,0,511,2025,4048,3832,4462,7881,5502,5250,0
April,19,678,2243,3773,3991,4458,7219,5083,4381,0
May,74,760,2288,3962,4207,4626,6776,5157,4804,0
June,75,1093,2294,3927,3853,5066,6698,4904,4504,0
July,78,1067,2368,3664,3876,10695,5822,4308,4219,0
August,115,967,2320,3521,3514,7315,5546,4346,4355,0
September,348,1028,2151,3042,3699,6267,5773,4359,4100,0


,Success_Rate_Pct
Launch_Month_Name,
January,34.6
February,37.5
March,38.2
April,37.8
May,36.9
June,36.3
July,32.4
August,33.8
September,36.5


**Description:** I extracted the **Launch_Year, Launch_Month, and an ordered Launch_Month_Name** from the Launched column, using a Categorical so months sort Jan to Dec. Counted projects by month by the year (months as rows, years as columns, zeros for empty combos). Then dropped Live projects and computed the success rate per month, pooling all years. This transformation's significance shows that when a project launches and it then relates to how it performs. The count table shows volume (which months/years are busiest); the success rate shows performance (which months actually succeed).

### **T3. String standardization: Strip hidden whitespace in Country and Name**

In [104]:
rows = []
for col in ["Country", "Name"]:
    b, a = df[col], df_transformed[col]
    rows.append({
        "Column": col,
        "Stray-whitespace rows (before)": (b != b.str.strip()).sum(),
        "Stray-whitespace rows (after)":  (a != a.str.strip()).sum(),
        "Rows changed":                   (b != a).sum(),
        "Unique values (before)":         b.nunique(),
        "Unique values (after)":          a.nunique(),
        "Repeated values (before)":       b.duplicated().sum(),
        "Repeated values (after)":        a.duplicated().sum(),
    })
display(pd.DataFrame(rows).set_index("Column").T)

# Table 2: the affected Country value, shown with repr() so the space is visible
country_evidence = pd.concat(
    {"Before": df["Country"].map(repr).value_counts(),
     "After":  df_transformed["Country"].map(repr).value_counts()},
    axis=1,
).loc[lambda t: t.index.str.contains("Netherlands")]
display(country_evidence)

# T3. String standardization (run AFTER df_transformed = df.copy(), and don't re-run that copy afterwards)
for col in ["Country", "Name"]:
    df_transformed[col] = df_transformed[col].str.strip()

assert (df_transformed["Country"] == df_transformed["Country"].str.strip()).all()
assert (df_transformed["Name"] == df_transformed["Name"].str.strip()).all()

Column,Country,Name
Stray-whitespace rows (before),2868,1108
Stray-whitespace rows (after),2868,1108
Rows changed,0,0
Unique values (before),22,372061
Unique values (after),22,372061
Repeated values (before),374831,2792
Repeated values (after),374831,2792


,Before,After
Country,,
'Netherlands ',2868,2868


In [105]:
print("Final shape of df_transformed:", df_transformed.shape)   # compare with the 'before' shape you printed
print("Columns added by transformations:", [c for c in df_transformed.columns if c not in ["ID","Name","Category","Subcategory","Country","Launched","Deadline","Goal","Pledged","Backers","State"]])

Final shape of df_transformed: (374853, 15)
Columns added by transformations: ['campaign_days', 'Launch_Year', 'Launch_Month', 'Launch_Month_Name']


### **Transformation purposes**
| # | Transformation | Column(s) affected | Analytical purpose |
|---|---|---|---|
| T1 | Reshaping: `pivot_table` (median Pledged by Category × State) | Category, State, Pledged | Compare typical pledges across categories and outcomes in one 15 × 5 table; median resists skew |
| T2 | Date/time extraction | Launched → Launch_Year, Launch_Month, Launch_Month_Name | Shows the trend analysis by month (success is lowest for July and December launches, about 32–33%, versus 38% in March) |
| T3 | String standardization | Country, Name | Removes hidden whitespace ('Netherlands ') so groupby and merge keys match exactly |

## **Integration (Activity 4B)**

**Datasets considered:**  
The brief asks for a second related dataset only if a defensible relationship exists. For Kickstarter data I considered:  
1. Exchange-rate tables, which would relate to Goal and Pledged through currency and date, but there are no existing datasets that align with my dataset, so there is no defensible key;
2. I also tried to find for US Census data, which needs a state or county that the file does not contain.  
  
**None of these is both available and defensible, so no external merge was performed.**

## **Cleaning (Activity 5)**

In [119]:
df_clean = df_transformed.copy()

# 1. Strings: strip whitespace in EVERY text column (you only checked Country and Name; Subcategory was never inspected)
text_cols = df_clean.select_dtypes(exclude=["number", "datetime", "category"]).columns
for c in text_cols:
    df_clean[c] = df_clean[c].str.strip()
print("Text columns still containing stray spaces:",
      {c: int((df_clean[c] != df_clean[c].str.strip()).sum()) for c in text_cols})

Text columns still containing stray spaces: {'Name': 0, 'Category': 0, 'Subcategory': 0, 'Country': 0, 'State': 0}


In [120]:
# 2. Duplicates (evidence for the 'Duplicate records' row)
print("Fully duplicated rows:        ", df_clean.duplicated().sum())
print("Duplicated IDs:               ", df_clean["ID"].duplicated().sum())
print("Repeated Name+Category+Goal+Deadline:", df_clean.duplicated(subset=["Name", "Category", "Goal", "Deadline"]).sum())
# Decision: keep all rows (different IDs / launch times = relaunches, not copies)

Fully duplicated rows:         0
Duplicated IDs:                0
Repeated Name+Category+Goal+Deadline: 16


In [121]:
# 3. Invalid values: Goal = 0 is impossible -> set to missing, keep the row
n_bad_goal = (df_clean["Goal"] == 0).sum()
df_clean["Goal"] = df_clean["Goal"].mask(df_clean["Goal"] == 0).astype("Int64")   # nullable integer allows <NA>
print("Goal set to missing:", n_bad_goal, "| missing Goal now:", df_clean["Goal"].isna().sum())

# 4. Data types
for c in ["Category", "Subcategory", "Country", "State"]:
    df_clean[c] = df_clean[c].astype("category")
print(df_clean.dtypes)

Goal set to missing: 4 | missing Goal now: 4
ID                            int64
Name                            str
Category                   category
Subcategory                category
Country                    category
Launched             datetime64[us]
Deadline             datetime64[us]
Goal                          Int64
Pledged                       int64
Backers                       int64
State                      category
campaign_days                 int64
Launch_Year                   int32
Launch_Month                  int32
Launch_Month_Name          category
dtype: object


In [122]:
# 5. Flags 
df_clean["Is_Final_State"] = df_clean["State"].ne("Live")

reached_goal = (df_clean["Pledged"] >= df_clean["Goal"]).fillna(False).astype(bool)
has_goal = df_clean["Goal"].notna()
df_clean["flag_state_pledge_conflict"] = (
    ((df_clean["State"] == "Failed") & reached_goal) |
    ((df_clean["State"] == "Successful") & ~reached_goal & has_goal)
)
df_clean["flag_backers_pledge_mismatch"] = (
    ((df_clean["Backers"] == 0) & (df_clean["Pledged"] > 0)) |
    ((df_clean["Backers"] > 0) & (df_clean["Pledged"] == 0))
)
print("State/pledge conflicts (expect 11 = 9 Failed + 2 Successful):", df_clean["flag_state_pledge_conflict"].sum())
print("Backers/pledge mismatches (expect 12 = 3 + 9):               ", df_clean["flag_backers_pledge_mismatch"].sum())
print("Canceled/Suspended that reached the goal (NOT flagged):       ",
      (df_clean["State"].isin(["Canceled", "Suspended"]) & reached_goal).sum())

State/pledge conflicts (expect 11 = 9 Failed + 2 Successful): 11
Backers/pledge mismatches (expect 12 = 3 + 9):                12
Canceled/Suspended that reached the goal (NOT flagged):        989


In [123]:
# 6. Date/time checks
snapshot = df_clean["Launched"].max()                       # latest launch in the file = approximate snapshot date
df_clean["flag_live_deadline_passed"] = (df_clean["State"] == "Live") & (df_clean["Deadline"] < snapshot.normalize())
print("Snapshot (latest Launched):", snapshot)
print("Deadline before Launched date:", (df_clean["Deadline"] < df_clean["Launched"].dt.normalize()).sum())
print("campaign_days min/max:", df_clean["campaign_days"].min(), df_clean["campaign_days"].max())
print("Live rows whose deadline already passed:", df_clean["flag_live_deadline_passed"].sum())
print("Projects launched in 2018 (partial year):", (df_clean["Launched"].dt.year == 2018).sum())

Snapshot (latest Launched): 2018-01-02 15:02:31
Deadline before Launched date: 0
campaign_days min/max: 1 92
Live rows whose deadline already passed: 7
Projects launched in 2018 (partial year): 124


#### **Cleaned Dataset**

In [124]:
df_clean.to_csv("kickstart_cleaned_dataset.csv", index=False)
print("Saved:", df_clean.shape)

Saved: (374853, 19)


This cleaned dataset is intended for analyzing which factors (category, goal size, launch timing) are associated with Kickstarter project success. Live projects are kept but excluded from success-rate analysis through Is_Final_State.

#### **Decision Record**  
| Issue | Evidence | Decision | Action (code) | Reason |
|---|---|---|---|---|
| Missing values | 0 missing cells in all 11 columns | None to impute; the 4 invalid goals become missing (see invalid values) | isna().sum() | Nothing to fill; imputing a goal would invent data |
| Duplicate records | 0 duplicated rows, 0 duplicated IDs; 2,792 repeated names (2,801 once whitespace is stripped); 16 rows repeat Name + Category + Goal + Deadline | Keep all rows | duplicated() checks | Different IDs and launch times indicate relaunches or resubmissions, not copies |
| Incorrect data types | Launched/Deadline load as text without parsing; categoricals stored as strings; Goal must allow missing | Parse dates, convert to category, Goal to nullable Int64 | parse_dates, astype | Correct operations and lower memory |
| Inconsistent categorical values | 'Netherlands ' (2,868 rows); 1,108 names with stray spaces | Strip whitespace; leave letter case and non-ASCII names as they are | .str.strip() | Whitespace is noise; emojis and accents are real content |
| Invalid / impossible values | 4 projects with Goal = 0 | Set Goal to missing, keep the rows | Goal.mask(Goal == 0) | A goal cannot be 0; the pledges and states are still usable |
| State vs pledge conflicts | 9 Failed with Pledged ≥ Goal; 2 Successful with Pledged < Goal; 989 Canceled/Suspended reached goal | Flag the 11 contradictory rows; do not flag the 989 | flag_state_pledge_conflict | Cancelling after reaching the goal is legitimate; Failed or Successful contradicting the amounts is suspicious |
| Backer/pledge mismatch | 3 rows with 0 backers but a pledge; 9 rows with backers but 0 pledge | Flag, keep | flag_backers_pledge_mismatch | Small counts; cannot tell which field is wrong |
| Date/time inconsistencies | Deadline never precedes Launched; durations 1–92 days; 7 Live projects have a deadline before the snapshot date; only 124 projects launched in 2018 | Flag stale Live rows; keep 2018 but note it is a partial year | flag_live_deadline_passed | State may be out of date; 2018 should not be read as a full year |
| Live projects | 2,798 rows with no final outcome | Keep, added Is_Final_State | Is_Final_State | Excluded from success-rate analysis by filter, not deleted |

### Validation

In [125]:
expected_cols = ["ID","Name","Category","Subcategory","Country","Launched","Deadline","Goal","Pledged","Backers","State"]
valid_goal = df_clean["Goal"].dropna()

checks = {
    "Expected columns are present":                    set(expected_cols).issubset(df_clean.columns),
    "Important columns have appropriate data types":   (pd.api.types.is_datetime64_any_dtype(df_clean["Launched"])
                                                        and pd.api.types.is_datetime64_any_dtype(df_clean["Deadline"])
                                                        and pd.api.types.is_numeric_dtype(df_clean["Goal"])
                                                        and pd.api.types.is_numeric_dtype(df_clean["Pledged"])
                                                        and pd.api.types.is_numeric_dtype(df_clean["Backers"])),
    "Required identifiers are not duplicated (ID)":    df_clean["ID"].is_unique,
    "Categorical values are consistent (no stray spaces)": all(
        (df_clean[c].astype(str) == df_clean[c].astype(str).str.strip()).all() for c in ["Category","Subcategory","Country","State"]),
    "Numerical values satisfy domain constraints":     bool((valid_goal > 0).all() and (df_clean["Pledged"] >= 0).all() and (df_clean["Backers"] >= 0).all()),
    "Date/time values are valid (Deadline not before launch date)": bool((df_clean["Deadline"] >= df_clean["Launched"].dt.normalize()).all()),
    "Final row count matches the raw file (no rows dropped)": len(df_clean) == len(df),
}
report = pd.DataFrame({"Check": checks.keys(), "Passed": checks.values()})
display(report)
assert report["Passed"].all(), "A validation check failed"
print(f"Final row count: {len(df_clean):,} rows x {df_clean.shape[1]} columns")

,Check,Passed
0,Expected columns are present,True
1,Important columns have appropriate data types,True
2,Required identifiers are not duplicated (ID),True
3,Categorical values are consistent (no stray spaces),True
4,Numerical values satisfy domain constraints,True
5,Date/time values are valid (Deadline not before launch date),True
6,Final row count matches the raw file (no rows dropped),True


Final row count: 374,853 rows x 19 columns


## **Final Reflection**

**1. Which data-wrangling step required the most reasoning rather than coding?**  
The additional checks that were added for further investigation needed more reasoning because i had to analyze what to do with the unusual rows, like deciding whether a Canceled project that reached its goal, a Failed project with Pledged >= Goal, or a Goal of 0 was an error. Eventually, I kept the 989 Canceled/Suspended projects (cancelling after reaching a goal is legitimate), flagged the 11 contradictory Failed/Successful rows, and set the 4 impossible goals to missing instead of deleting the rows.

**2. Which NumPy operation was most useful in your project?**  
The most useful NumPy operation in my project was **np.inf**. In **pd.cut**, the bin edges are the boundary numbers that separate the goal groups (for example 0, 1,000, 5,000, and so on). I used np.inf, which means infinity, as the last bin edge, so the final group, “Over $100k”, has no upper limit. This way every project with a goal above 100,000 USD is included, even very large ones, and no project is left out of the groups. Without it I would have had to choose a maximum number, and any goal above it would become missing (NaN).

**3. Which Pandas operation was most useful?**  
Using the **groupby().agg()** made data aggregation and validation much easier because this operation produced project counts, success rate, average backers, and median pledged by category, year, or goal tier.

**4. What would be risky about analyzing the raw dataset without cleaning and validation?**  
Five data-quality issues need handling before analysis. First, the 2,798 live projects have no final outcome, so counting them as non-successes would understate success rates. Second, the country value 'Netherlands ' carries a trailing space and would be treated as distinct from 'Netherlands' in grouping or merging. Third, Goal = 0 makes percent-to-goal undefined due to division by zero. Fourth, extreme values—goals reaching about 166 million against a mean of 45,864 versus a median of just 5,500—would distort means. Finally, 2018 is a partial year, so its 0.0% success rate would read as a real collapse rather than an artifact of incomplete data.  

**5. What additional data would improve the analysis?**  
Maybe the date the data was collected, which justifies Live projects.

**6. If this were a real organizational project, what would you document or automate next?**  
If it were a real organizational project, i should create an script that could read the raw CSV and write the cleaned version. And for every addition of new data, there should be a way to automatically run validation checks.
